<a href="https://colab.research.google.com/github/BernardoBremer/Human_Following_Robot/blob/main/2_1_evaluation_sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metrics and Evaluation - California Housing Dataset

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [17]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)
print(x.shape)
x.head()

(20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [18]:
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

Train: (16512, 8), Val: (2064, 8), Test: (2064, 8)


In [24]:
feature_cols = list(x.columns)
geo_cols = ["latitude", "ongitude"]
non_geo_idx = [i for i, c in enumerate(feature_cols) if c not in geo_cols]
geo_idx = [i for i, c in enumerate(feature_cols) if c in geo_cols]

x_train_t = torch.tensor(x_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
x_val_t = torch.tensor(x_val.values, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
x_test_t = torch.tensor(x_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)



In [25]:
class Preprocessor(nn.Module):
    def __init__(self, non_geo_idx, geo_idx, lower_bound, upper_bound, mean, std):
        super().__init__()
        self.non_geo_idx = non_geo_idx
        self.geo_idx = geo_idx
        self.register_buffer("lower", lower_bound)
        self.register_buffer("upper", upper_bound)
        self.register_buffer("mean", mean)
        self.register_buffer("std", std)

    def forward(self, x):
        non_geo = x[:, self.non_geo_idx]
        geo = x[:, self.geo_idx]
        clamped = torch.clamp(non_geo, min=self.lower, max=self.upper)
        scaled = (clamped - self.mean) / self.std
        return torch.cat([scaled, geo], dim=1)


class HousingModel(nn.Module):
    def __init__(self, preprocessor, hidden_sizes, dropout=0.0):
        super().__init__()
        self.preprocessor = preprocessor
        layers = []
        input_dim = 8
        for h in hidden_sizes:
            layers.append(nn.Linear(input_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = self.preprocessor(x)
        return self.net(x)

In [26]:
def train_model(model, x_train, y_train, x_val, y_val, epochs=200, lr=0.001, batch_size=256):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    train_ds = TensorDataset(x_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            pred = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        train_losses.append(epoch_loss / len(train_ds))

        model.eval()
        with torch.no_grad():
            val_pred = model(x_val)
            val_loss = criterion(val_pred, y_val).item()
            val_losses.append(val_loss)

        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_losses[-1]:.4f} - Val Loss: {val_loss:.4f}")

    return train_losses, val_losses

In [27]:
def evaluate(model, x, y):
    model.eval()
    with torch.no_grad():
        preds = model(x).squeeze()
        targets = y.squeeze()
        mae = torch.mean(torch.abs(preds - targets)).item()
        mse = torch.mean((preds - targets) ** 2).item()
        rmse = mse ** 0.5
        ss_res = torch.sum((targets - preds) ** 2).item()
        ss_tot = torch.sum((targets - targets.mean()) ** 2).item()
        r2 = 1 - ss_res / ss_tot
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(10, 15))

for ax, col in zip(axs.ravel(), x_train.columns):
    sns.scatterplot(x=x_train[col], y=y, ax=ax)

plt.tight_layout()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

preprocessing = make_pipeline(
    ColumnTransformer(transformers=[
        ("num", MinMaxScaler(), x.columns)
    ])
)

In [ ]:
x_train_processed = preprocessing.fit_transform(x_train)
x_test_processed = preprocessing.transform(x_test)
x_val_processed = preprocessing.transform(x_val)

In [ ]:
x_train_processed

## Model Training and Evaluation

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x_train_processed, y_train)
model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error


def evaluate_model(model, x, y):
    y_pred = model.predict(x)
    mae = mean_absolute_error(y, y_pred)
    mse = mean_squared_error(y, y_pred)
    rmse = root_mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)

    return {"MAE": mae, "MSE": mse, "R²": r2, "RMSE": rmse}

In [ ]:
import pandas as pd

train_metrics = evaluate_model(model, x_train_processed, y_train)
test_metrics = evaluate_model(model, x_test_processed, y_test)

metrics_df = pd.DataFrame({"Train": train_metrics, "Test": test_metrics})
metrics_df.T

In [ ]:
from sklearn.neural_network import MLPRegressor

model = MLPRegressor(hidden_layer_sizes=(50, 50), solver="adam")

model.fit(x_train_processed, y_train)

train_metrics = evaluate_model(model, x_train_processed, y_train)
test_metrics = evaluate_model(model, x_test_processed, y_test)

metrics_df = pd.DataFrame({"Train": train_metrics, "Test": test_metrics})
metrics_df.T

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (50, 50)],
    "solver": ["adam", "lbfgs"],
    "alpha": [0.0001, 0.001, 0.01],
}

grid_search = GridSearchCV(
    MLPRegressor(),
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",)

In [ ]:
grid_search.fit(x_train_processed, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
model = grid_search.best_estimator_

train_metrics = evaluate_model(model, x_train_processed, y_train)
test_metrics = evaluate_model(model, x_test_processed, y_test)

metrics_df = pd.DataFrame({"Train": train_metrics, "Test": test_metrics})
metrics_df.T

In [ ]:
val_metrics = evaluate_model(model, x_val_processed, y_val)
val_metrics